In [0]:
spark.sql("CREATE DATABASE IF NOT EXISTS workspace.gold COMMENT 'Capa gold'")

In [0]:
#spark.sql("DROP TABLE IF EXISTS workspace.gold.dim_productos")

In [0]:
spark.sql("""
CREATE TABLE IF NOT EXISTS workspace.gold.dim_productos (
  id_producto LONG,
  nombre_producto STRING
)
""")


In [0]:
import pandas as pd

df = spark.table("workspace.silver.adidas_us_sales").toPandas()

# Se crea la dimension de tiendas, desde la tabla de ventas
dim_productos = df[['producto']].drop_duplicates().reset_index(drop=True)
dim_productos['id_producto'] = dim_productos.index + 1
dim_productos = dim_productos.rename(columns={'producto': 'nombre_producto'})
dim_productos = dim_productos[['id_producto', 'nombre_producto']]

df_spark = spark.createDataFrame(dim_productos)


In [0]:

# Usamos overwrite para reemplazar completamente los datos existentes
df_spark.write.format("delta") \
    .option("mergeSchema", "true") \
    .mode("overwrite") \
    .saveAsTable("workspace.gold.dim_productos")